# Blender Kitchen Scene Viewer

Interaktywny viewer scen 3D kuchni z HD-EPIC Digital Twin.  
Wczytuje mesze `.obj` z archiwum `blenders.zip` dla uczestnika powiązanego z danym ID wideo.

In [ ]:
# ─── ARGUMENT: ustaw ID wideo tutaj ───────────────────────────────────────────
# VIDEO_ID = "P01-20240202-110250"
# VIDEO_ID = "P01-20240202-161948"
VIDEO_ID = "P01-20240202-110250"
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
from blender_viewer import *

In [ ]:
participant = participant_from_video_id(VIDEO_ID)
print(f"Video ID  : {VIDEO_ID}")
print(f"Participant: {participant}")
print(f"Loading meshes from: {BLENDERS_ZIP}")

In [ ]:
meshes = load_scene_meshes(BLENDERS_ZIP, participant)

In [ ]:
print(f"{'Object':40s}  {'Vertices':>10s}  {'Faces':>8s}")
print("-" * 64)
for m in meshes:
    print(f"{m['name']:40s}  {len(m['vertices']):>10,}  {len(m['faces']):>8,}")

In [ ]:
fig = build_figure(meshes, participant, VIDEO_ID)
fig.show()

In [ ]:
# ─── ARGUMENT: frame index to visualise ──────────────────────────────────────
FRAME_INDEX = 1761
# FRAME_INDEX = 230
# FRAME_INDEX = 682
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
frame = load_frame_data(VIDEO_ID, FRAME_INDEX)
if frame is None:
    raise ValueError(f"Frame {FRAME_INDEX} not found in {VIDEO_ID}")

T = np.array(frame["T_world_device"])
print(f"Frame         : {frame['frame_index']}")
print(f"Camera pos    : {T[:, 3]}")
print(f"Gaze (pixels) : {frame.get('gaze_centre_in_pixels')}")
print(f"Gaze (world)  : {frame.get('gaze_direction_in_world')}")

In [ ]:
hand_poses = load_hand_poses_for_frame(VIDEO_ID, frame)
print(f"Left  hand confidence : {hand_poses['left_confidence']:.3f}  "
      f"({'tracked' if hand_poses['left_wrist'] is not None else 'not tracked'})")
print(f"Right hand confidence : {hand_poses['right_confidence']:.3f}  "
      f"({'tracked' if hand_poses['right_wrist'] is not None else 'not tracked'})")

fig = build_figure(meshes, participant, VIDEO_ID)
add_camera_pose_to_figure(fig, frame, axis_len=0.15)
add_hands_to_figure(fig, hand_poses)
fig.update_layout(title=dict(
    text=f"Camera Pose + Hands — {participant}  |  {VIDEO_ID}  |  frame {FRAME_INDEX}"
))
fig.show()

In [ ]:
slam_idx = slam_index_for_video(participant, VIDEO_ID)
print(f"SLAM segment index for {VIDEO_ID}: {slam_idx}")
print("Loading point cloud (may take ~10 s) ...")
pts = load_pointcloud(participant, slam_idx)
print(f"Loaded {len(pts):,} points after quality filtering and downsampling.")

In [ ]:
fig_pc = build_pointcloud_figure(pts, participant, VIDEO_ID, slam_idx)
fig_pc.show()

In [ ]:
render_depth_map(VIDEO_ID, frame, pts, max_depth=5.0)

In [ ]:
# ─── ARGUMENT: digital twin parameters ───────────────────────────────────────
TEXT_PROMPT  = "food processor"
TWIN_OUT_DIR = Path("/usr/prakt/s0021/EH/hd-epic-annotations-viewer/digital_twins")
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
work_dir, job_id = submit_twin_job(VIDEO_ID, FRAME_INDEX, TEXT_PROMPT, TWIN_OUT_DIR)
print(f"\nOutputs will appear in: {work_dir}")
print(f"Run the cell below once the job completes (check: squeue -j {job_id})")

In [ ]:
twin = load_twin_result(work_dir)
visualize_twin(twin, frame, VIDEO_ID, TEXT_PROMPT, meshes, participant, pts_world=pts)

In [ ]:
# # Overlay rendered geometric depth on the real photo — points shown in red.
# import numpy as np
# import matplotlib.pyplot as plt

# # ── 1. Render depth (no scatter plot) ───────────────────────────────────────
# depth_img, valid = render_depth_map(
#     VIDEO_ID, frame, pts, max_depth=5.0,
#     plot=False, return_maps=True,
# )

# # ── 2. Get the real photo ────────────────────────────────────────────────────
# rgb = np.asarray(twin["frame"], dtype=np.uint8)
# H_r, W_r = rgb.shape[:2]
# if depth_img.shape[:2] != (H_r, W_r):
#     raise ValueError(
#         f"Depth image shape {depth_img.shape[:2]} != photo shape {(H_r, W_r)}."
#     )

# # ── 3. Build red RGBA overlay ────────────────────────────────────────────────
# # Each projected point becomes a red pixel; alpha scales with depth coverage.
# overlay = np.zeros((H_r, W_r, 4), dtype=np.float32)
# overlay[valid, 0] = 1.0   # full red
# overlay[valid, 3] = 0.80  # mostly opaque where a point landed

# # ── 4. Three-panel figure ────────────────────────────────────────────────────
# fig, axs = plt.subplots(1, 3, figsize=(18, 6), facecolor="#0a0a15")
# for ax in axs:
#     ax.set_facecolor("black")
#     ax.axis("off")

# axs[0].imshow(rgb)
# axs[0].set_title("Real photo", color="#ccccdd", fontsize=12)

# axs[1].imshow(rgb)
# axs[1].imshow(overlay)
# axs[1].set_title("Red = projected depth points", color="#ccccdd", fontsize=12)

# # Third panel: zoom into the object bbox if available
# axs[2].imshow(rgb)
# axs[2].imshow(overlay)
# if "box" in twin:
#     x1, y1, x2, y2 = (int(v) for v in twin["box"])
#     pad = 60
#     axs[2].set_xlim(max(0, x1 - pad), min(W_r, x2 + pad))
#     axs[2].set_ylim(min(H_r, y2 + pad), max(0, y1 - pad))
#     axs[2].set_title("Zoom on object (depth overlay)", color="#ccccdd", fontsize=12)
# else:
#     axs[2].set_title("Depth overlay (no bbox found)", color="#ccccdd", fontsize=12)

# plt.suptitle(
#     f"{VIDEO_ID}  |  frame {frame['frame_index']}  —  depth coverage: {valid.mean():.1%} of pixels",
#     color="#aaaacc", fontsize=11, y=1.01,
# )
# plt.tight_layout()
# plt.show()

# # ── 5. Save the middle panel (photo + red depth overlay) as a high-res PNG ──
# from PIL import Image as _PILImage

# _save_path = work_dir / f"depth_overlay_{frame['frame_index']}.png"

# # Composite: RGB base + red overlay at full pixel resolution (no matplotlib scaling)
# _base = rgb.astype(np.float32)
# _r, _a = overlay[..., 0], overlay[..., 3]            # red channel + alpha
# _comp = _base.copy()
# _comp[..., 0] = np.clip(_base[..., 0] * (1 - _a) + _r * 255 * _a, 0, 255)
# _comp[..., 1] = np.clip(_base[..., 1] * (1 - _a),                  0, 255)
# _comp[..., 2] = np.clip(_base[..., 2] * (1 - _a),                  0, 255)
# _comp = _comp.astype(np.uint8)

# _PILImage.fromarray(_comp).save(_save_path)
# print(f"Saved high-res overlay → {_save_path}  ({W_r}×{H_r} px)")


In [ ]:
fig_cmp = build_slam_vs_sam3d_figure(
    twin, frame, VIDEO_ID, pts, meshes, participant,
    mask_dilation=60,
)
fig_cmp.show()